# 🚬 Impact of Vaping on Workplace Productivity
### A Data-Driven Analysis | Survey n = 146

---

**Objetivo:** Cuantificar el impacto del vaping en la productividad laboral/escolar a través de análisis estadístico, modelado de pérdida de tiempo y segmentación de usuarios.

**Estructura del notebook:**
1. Setup & Carga de datos
2. Limpieza y normalización
3. Análisis exploratorio (EDA)
4. Análisis de productividad
5. Modelo de pérdida de tiempo
6. Pruebas estadísticas
7. Segmentación de usuarios
8. Dashboard consolidado
9. Conclusiones

## 1. Setup & Imports

In [ ]:
# ─────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# THEME GLOBAL
# ─────────────────────────────────────────────
PALETTE   = px.colors.qualitative.Vivid
BG_COLOR  = '#0f0f0f'
FONT_COLOR = '#f0f0f0'
ACCENT    = '#7f5af0'

BASE_LAYOUT = dict(
    template='plotly_dark',
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#16213e',
    font=dict(family='Inter, sans-serif', color=FONT_COLOR),
    title_font_size=18,
    margin=dict(t=60, b=40, l=40, r=40)
)

print('✅ Setup completo')

## 2. Carga & Limpieza de Datos

In [ ]:
# ─────────────────────────────────────────────
# CARGA  (ajusta la ruta si es necesario)
# ─────────────────────────────────────────────
RAW_PATH = 'Vape__Analisis_de_datos.csv'

df_raw = pd.read_csv(RAW_PATH, encoding='latin1')

# Eliminar filas completamente vacías (artefacto del export de Google Forms)
df_raw = df_raw.dropna(how='all').reset_index(drop=True)

# ─────────────────────────────────────────────
# RENOMBRAR COLUMNAS → alias cortos
# ─────────────────────────────────────────────
COLUMN_MAP = {
    df_raw.columns[0]:  'fecha',
    df_raw.columns[1]:  'duracion_vape',
    df_raw.columns[2]:  'patron_uso',
    df_raw.columns[3]:  'hits_por_periodo',
    df_raw.columns[4]:  'duracion_periodo',
    df_raw.columns[5]:  'afecta_concentracion',
    df_raw.columns[6]:  'productividad_score',
    df_raw.columns[7]:  'procrastinacion',
    df_raw.columns[8]:  'uso_trabajo_escuela',
    df_raw.columns[9]:  'afecta_calidad',
    df_raw.columns[10]: 'efecto_vape',
    df_raw.columns[11]: 'feedback_externo'
}

df = df_raw.rename(columns=COLUMN_MAP).copy()

# ─────────────────────────────────────────────
# NORMALIZACIÓN
# ─────────────────────────────────────────────
# Asegurar tipo numérico en productividad
df['productividad_score'] = pd.to_numeric(df['productividad_score'], errors='coerce')

# Convertir fecha
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

# Strip espacios en columnas categóricas
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].apply(lambda c: c.str.strip().str.lower())

# ─────────────────────────────────────────────
# ORDEN ORDINAL para variables de duración
# ─────────────────────────────────────────────
ORDER_DURACION = ['1 a 3 dias', '4 a 7 dias', '7 a 14 dias', '3 semanas', 'mas de un mes']
ORDER_HITS     = ['1 a 3', '4 a 7', '8 a 11', '12 a 16', 'mas de 16']
ORDER_PERIODO  = ['1 a 3 segundos', '4 a 7 segundos', '7 a 11 segundos',
                  '11 a 15 segundos', '15 a 30 segundos', '30 a 60 segundos',
                  '1 a 3 minutos', '3 a 5 minutos', 'mas de 5 minutos']

df['duracion_vape']   = pd.Categorical(df['duracion_vape'],   categories=ORDER_DURACION, ordered=True)
df['hits_por_periodo'] = pd.Categorical(df['hits_por_periodo'], categories=ORDER_HITS,     ordered=True)
df['duracion_periodo'] = pd.Categorical(df['duracion_periodo'], categories=ORDER_PERIODO,  ordered=True)

# ─────────────────────────────────────────────
# RESUMEN
# ─────────────────────────────────────────────
print(f'📊 Registros válidos : {len(df)}')
print(f'📅 Rango de fechas   : {df["fecha"].min().date()} → {df["fecha"].max().date()}')
print(f'\n🔍 Vista general:')
df.head(5)

In [ ]:
# Estadísticos descriptivos de la columna numérica
df[['productividad_score']].describe().round(2)

## 3. Análisis Exploratorio (EDA)

In [ ]:
def plot_bar(series: pd.Series, title: str, x_label: str = '', 
             order: list = None, color_seq: list = PALETTE) -> go.Figure:
    """
    Genera un gráfico de barras horizontal con Plotly.

    Parameters
    ----------
    series     : pd.Series  — datos categóricos
    title      : str        — título del gráfico
    x_label    : str        — etiqueta del eje X
    order      : list       — orden personalizado de categorías
    color_seq  : list       — paleta de colores

    Returns
    -------
    go.Figure
    """
    counts = series.value_counts()
    if order:
        counts = counts.reindex([o for o in order if o in counts.index])
    pcts = (counts / counts.sum() * 100).round(1)

    fig = go.Figure(go.Bar(
        y=counts.index.tolist(),
        x=counts.values,
        orientation='h',
        text=[f'{v}  ({p}%)' for v, p in zip(counts.values, pcts)],
        textposition='outside',
        marker=dict(color=color_seq[:len(counts)],
                    line=dict(width=0)),
    ))
    fig.update_layout(**BASE_LAYOUT, title=title,
                      xaxis_title=x_label or 'Frecuencia',
                      height=350)
    return fig


# ── Duración del vape ──────────────────────────────────────────────────────
fig1 = plot_bar(df['duracion_vape'], 
                '¿Cuánto tiempo dura un vape (4 000 puffs)?',
                order=ORDER_DURACION[::-1])
fig1.show()

# ── Efecto principal ──────────────────────────────────────────────────────
fig2 = plot_bar(df['efecto_vape'], 'Efecto principal reportado del vape')
fig2.show()

# ── Patrón de uso ─────────────────────────────────────────────────────────
vc = df['patron_uso'].value_counts()
fig3 = px.pie(
    values=vc.values, names=vc.index,
    title='Patrón de uso: continuo vs periódico',
    color_discrete_sequence=PALETTE,
    hole=0.4
)
fig3.update_layout(**BASE_LAYOUT)
fig3.show()

## 4. Análisis de Productividad

In [ ]:
# ─────────────────────────────────────────────
# 4.1 Distribución del score de productividad
# ─────────────────────────────────────────────
prod = df['productividad_score'].dropna()

mean_val   = prod.mean()
median_val = prod.median()
mode_val   = prod.mode()[0]

print(f'📈 Media   : {mean_val:.2f} / 10  →  {mean_val*10:.1f}% de productividad')
print(f'📊 Mediana : {median_val:.1f} / 10')
print(f'🏆 Moda    : {mode_val:.0f} / 10')

fig4 = go.Figure()
fig4.add_trace(go.Histogram(
    x=prod, nbinsx=10,
    marker_color=ACCENT, opacity=0.85,
    name='Distribución'
))
for val, label, color in [(mean_val, f'Media {mean_val:.1f}', '#ff6b6b'),
                           (median_val, f'Mediana {median_val:.0f}', '#f7b731'),
                           (mode_val,   f'Moda {mode_val:.0f}',   '#45b7d1')]:
    fig4.add_vline(x=val, line_dash='dash', line_color=color, line_width=2,
                   annotation_text=label, annotation_position='top')

fig4.update_layout(**BASE_LAYOUT,
                   title='Distribución: Score de Productividad (escala 1-10)',
                   xaxis_title='Productividad percibida', yaxis_title='Frecuencia',
                   height=400)
fig4.show()

In [ ]:
# ─────────────────────────────────────────────
# 4.2 Productividad por efecto del vape
# ─────────────────────────────────────────────
prod_by_effect = (
    df.groupby('efecto_vape')['productividad_score']
    .agg(['mean', 'median', 'std', 'count'])
    .rename(columns={'mean':'Media','median':'Mediana','std':'DE','count':'n'})
    .sort_values('Media')
    .reset_index()
)

fig5 = go.Figure()
fig5.add_trace(go.Bar(
    x=prod_by_effect['efecto_vape'], y=prod_by_effect['Media'],
    error_y=dict(type='data', array=prod_by_effect['DE'], visible=True),
    text=prod_by_effect['Media'].round(2),
    textposition='outside',
    marker_color=PALETTE[:len(prod_by_effect)],
    name='Media ± DE'
))
fig5.add_hline(y=mean_val, line_dash='dot', line_color='#ff6b6b',
               annotation_text=f'Media global {mean_val:.1f}')
fig5.update_layout(**BASE_LAYOUT,
                   title='Productividad Media por Efecto Reportado del Vape',
                   yaxis=dict(range=[0, 12]),
                   height=420)
fig5.show()
print(prod_by_effect[['efecto_vape','Media','Mediana','DE','n']].to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# 4.3 Autoconciencia: productividad percibida
#     vs impacto real en concentración
# ─────────────────────────────────────────────
cross_conc = pd.crosstab(df['afecta_concentracion'],
                          df['uso_trabajo_escuela'],
                          normalize='index').round(3) * 100

fig6 = px.imshow(cross_conc,
                 text_auto='.1f',
                 color_continuous_scale='Viridis',
                 labels=dict(x='¿Usa en trabajo/escuela?',
                             y='¿Afecta concentración?',
                             color='% de grupo'),
                 title='Heatmap: ¿Afecta concentración? × ¿Usa en trabajo/escuela? (% por fila)')
fig6.update_layout(**BASE_LAYOUT, height=350)
fig6.show()

In [ ]:
# ─────────────────────────────────────────────
# 4.4 Box plots: score por variables clave
# ─────────────────────────────────────────────
cols_to_compare = [
    ('afecta_concentracion', '¿Afecta concentración?'),
    ('uso_trabajo_escuela',   '¿Usa en trabajo/escuela?'),
    ('efecto_vape',           'Efecto del vape'),
]

fig7 = make_subplots(rows=1, cols=3,
                     subplot_titles=[c[1] for c in cols_to_compare])

for idx, (col, label) in enumerate(cols_to_compare, start=1):
    for cat in df[col].dropna().unique():
        subset = df.loc[df[col] == cat, 'productividad_score'].dropna()
        fig7.add_trace(go.Box(y=subset, name=cat, showlegend=(idx == 1)), row=1, col=idx)

fig7.update_layout(**BASE_LAYOUT,
                   title='Distribución de Productividad por Variable Categórica',
                   height=420)
fig7.show()

## 5. Modelo de Pérdida de Tiempo

In [ ]:
# ─────────────────────────────────────────────
# CONSTANTES DEL MODELO
# ─────────────────────────────────────────────
PUFFS_PER_VAPE     = 4_000   # puffs totales en un vape estándar
LABOR_HOURS        = 8       # horas de jornada laboral
LABOR_SECONDS      = LABOR_HOURS * 3_600
LABOR_FRACTION     = 1/3     # fracción del día que corresponde a jornada

# Tiempo de recuperación (pausa entre hits) — segundos
REFRACTORY_SECONDS = 80

# Hits por puff (hits reales por acción de vapeo)
HITS_PER_ACTION    = 5

# ─────────────────────────────────────────────
# PROMEDIOS CALCULADOS DESDE LOS DATOS
# ─────────────────────────────────────────────

def weighted_mean(series: pd.Series, value_map: dict) -> float:
    """
    Calcula la media ponderada de una variable categórica ordinal
    mapeando cada categoría a su valor representativo.

    Parameters
    ----------
    series    : pd.Series  — columna categórica del DataFrame
    value_map : dict       — {categoría: valor_numérico_representativo}

    Returns
    -------
    float — media ponderada
    """
    counts = series.value_counts()
    total  = 0.0
    n      = 0
    for cat, val in value_map.items():
        freq   = counts.get(cat, 0)
        total += freq * val
        n     += freq
    return total / n if n > 0 else 0.0


# Hits promedio por periodo
HITS_MAP = {'1 a 3': 2.0, '4 a 7': 5.5, '8 a 11': 9.5,
            '12 a 16': 14.0, 'mas de 16': 18.0}
avg_hits = weighted_mean(df['hits_por_periodo'], HITS_MAP)

# Duración promedio de cada periodo de uso (segundos)
PERIOD_MAP = {
    '1 a 3 segundos': 2.0,   '4 a 7 segundos': 5.5,
    '7 a 11 segundos': 9.0,  '11 a 15 segundos': 13.0,
    '15 a 30 segundos': 22.5,'30 a 60 segundos': 45.0,
    '1 a 3 minutos': 120.0,  '3 a 5 minutos': 240.0,
    'mas de 5 minutos': 420.0
}
avg_lapse = weighted_mean(df['duracion_periodo'], PERIOD_MAP)

# Vida promedio del vape (días)
DURATION_MAP = {'1 a 3 dias': 2.0, '4 a 7 dias': 5.5,
                '7 a 14 dias': 10.5, '3 semanas': 21.0,
                'mas de un mes': 45.0}
avg_lifetime_days = weighted_mean(df['duracion_vape'], DURATION_MAP)

print(f'Hits promedio por periodo  : {avg_hits:.2f}')
print(f'Duración promedio de lapse : {avg_lapse:.2f} seg')
print(f'Vida promedio del vape     : {avg_lifetime_days:.2f} días')

In [ ]:
# ─────────────────────────────────────────────
# CÁLCULO PRINCIPAL
# ─────────────────────────────────────────────

def time_loss_model(puffs: int, avg_hits: float, avg_lapse: float,
                    avg_lifetime_days: float, refractory: float,
                    labor_fraction: float, labor_seconds: int) -> dict:
    """
    Estima el tiempo laboral perdido en vaping.

    Supuestos
    ---------
    - Un 'hit' equivale a un puff.
    - La jornada laboral representa 1/3 del día.
    - Se mantiene el mismo patrón de uso en horas laborales.

    Returns
    -------
    dict con métricas calculadas
    """
    # ¿Cuántos periodos de vapeo por día completo?
    periods_per_day = (puffs / avg_hits) / avg_lifetime_days

    # Periodos durante jornada laboral
    periods_labor   = periods_per_day * labor_fraction

    # Tiempo perdido = (duración del lapse + recuperación) × periodos
    lost_seconds = (avg_lapse + refractory) * periods_labor
    lost_minutes = lost_seconds / 60

    # Porcentaje de jornada perdido
    pct_lost = (lost_seconds / labor_seconds) * 100

    # Productividad efectiva
    effective_labor_min = (labor_seconds - lost_seconds) / 60

    return {
        'periodos_por_dia'       : round(periods_per_day, 1),
        'periodos_en_jornada'    : round(periods_labor, 1),
        'tiempo_perdido_min'     : round(lost_minutes, 1),
        'tiempo_perdido_seg'     : round(lost_seconds, 1),
        'pct_jornada_perdido'    : round(pct_lost, 1),
        'tiempo_util_min'        : round(effective_labor_min, 1),
    }


results = time_loss_model(
    puffs            = PUFFS_PER_VAPE,
    avg_hits         = avg_hits,
    avg_lapse        = avg_lapse,
    avg_lifetime_days= avg_lifetime_days,
    refractory       = REFRACTORY_SECONDS,
    labor_fraction   = LABOR_FRACTION,
    labor_seconds    = LABOR_SECONDS
)

print('─' * 48)
print('     RESULTADOS DEL MODELO DE PÉRDIDA DE TIEMPO')
print('─' * 48)
for k, v in results.items():
    print(f'  {k:<30} {v}')
print('─' * 48)
print(f'\n  ⚠️  {results["pct_jornada_perdido"]}% de la jornada laboral se pierde en vapear')
print(f'  ⏱️  Equivale a {results["tiempo_perdido_min"]} minutos al día')

In [ ]:
# ─────────────────────────────────────────────
# ANÁLISIS DE SENSIBILIDAD
# ¿Qué pasa si la gente usa más o menos?
# ─────────────────────────────────────────────

lifetime_scenarios = [2, 5, 10.5, 21, 45]   # días
scenario_labels    = ['1–3 días', '4–7 días', '7–14 días', '3 semanas', '+1 mes']
pct_losses = []
min_losses = []

for ld in lifetime_scenarios:
    r = time_loss_model(
        puffs=PUFFS_PER_VAPE, avg_hits=avg_hits, avg_lapse=avg_lapse,
        avg_lifetime_days=ld, refractory=REFRACTORY_SECONDS,
        labor_fraction=LABOR_FRACTION, labor_seconds=LABOR_SECONDS
    )
    pct_losses.append(r['pct_jornada_perdido'])
    min_losses.append(r['tiempo_perdido_min'])

fig8 = make_subplots(rows=1, cols=2, subplot_titles=[
    '% de jornada perdido', 'Minutos perdidos al día'
])

for col, data, color in [(1, pct_losses, ACCENT), (2, min_losses, '#2cb67d')]:
    fig8.add_trace(go.Bar(
        x=scenario_labels, y=data,
        text=[f'{d:.0f}' for d in data], textposition='outside',
        marker_color=color
    ), row=1, col=col)

fig8.update_layout(**BASE_LAYOUT,
                   title='Análisis de Sensibilidad: Impacto según vida útil del vape',
                   height=420, showlegend=False)
fig8.show()

## 6. Pruebas Estadísticas

In [ ]:
# ─────────────────────────────────────────────
# 6.1 Chi-cuadrado entre pares de variables
#     Hipótesis nula: las variables son independientes
# ─────────────────────────────────────────────

def chi2_test(df: pd.DataFrame, col_a: str, col_b: str) -> dict:
    """
    Prueba de independencia chi² entre dos variables categóricas.

    Returns
    -------
    dict con estadístico, p-valor, grados de libertad y conclusión
    """
    table     = pd.crosstab(df[col_a], df[col_b])
    chi2, p, dof, expected = chi2_contingency(table)
    # Cramér's V — tamaño del efecto
    n = table.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))

    return {
        'Par'          : f'{col_a}  ×  {col_b}',
        'Chi²'         : round(chi2, 3),
        'p-valor'      : round(p, 4),
        'DoF'          : dof,
        "Cramér's V"   : round(cramers_v, 3),
        'Asociación'   : ('✅ Significativa' if p < 0.05 else '❌ No significativa')
    }


pairs = [
    ('efecto_vape',            'afecta_concentracion'),
    ('efecto_vape',            'uso_trabajo_escuela'),
    ('uso_trabajo_escuela',    'afecta_calidad'),
    ('afecta_concentracion',   'procrastinacion'),
    ('afecta_concentracion',   'afecta_calidad'),
    ('feedback_externo',       'afecta_calidad'),
]

results_chi2 = pd.DataFrame([chi2_test(df, a, b) for a, b in pairs])
print('Chi² — Pruebas de independencia entre variables')
print('=' * 70)
print(results_chi2.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# 6.2 Correlación de Spearman
#     Mide relación monotónica (no asume normalidad)
# ─────────────────────────────────────────────

# Codificación ordinal para análisis de correlación
encode_map = {
    'afecta_concentracion' : {'no': 0, 'tal vez': 1, 'si': 2},
    'afecta_calidad'       : {'no': 0, 'tal vez': 1, 'si': 2},
    'procrastinacion'      : {'no': 0, 'tal vez': 1, 'si': 2},
    'uso_trabajo_escuela'  : {'no': 0, 'si': 1},
    'feedback_externo'     : {'no': 0, 'si': 1},
    'efecto_vape'          : {'energia': 4, 'felicidad': 3, 'relajacion': 2, 'sueno': 1, 'mareo': 0},
    'duracion_vape'        : {'1 a 3 dias': 1, '4 a 7 dias': 2, '7 a 14 dias': 3,
                              '3 semanas': 4, 'mas de un mes': 5},
    'hits_por_periodo'     : {'1 a 3': 1, '4 a 7': 2, '8 a 11': 3, '12 a 16': 4, 'mas de 16': 5},
}

df_enc = df.copy()
for col, mapping in encode_map.items():
    df_enc[col] = df_enc[col].astype(str).map(mapping)

corr_cols = ['productividad_score'] + list(encode_map.keys())
df_corr   = df_enc[corr_cols].dropna()

spearman_matrix = df_corr.corr(method='spearman')

fig9 = px.imshow(
    spearman_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    zmin=-1, zmax=1,
    title='Matriz de Correlación de Spearman'
)
fig9.update_layout(**BASE_LAYOUT, height=550)
fig9.show()

# Ranking de correlaciones con productividad
corr_prod = (
    spearman_matrix['productividad_score']
    .drop('productividad_score')
    .abs()
    .sort_values(ascending=False)
)
print('\n📊 Variables más correlacionadas con productividad_score:')
print(corr_prod.to_string())

## 7. Segmentación de Usuarios

In [ ]:
# ─────────────────────────────────────────────
# 7.1 Heavy users vs Light users
#     Criterio: hits_por_periodo (≥8 = heavy)
# ─────────────────────────────────────────────
heavy_cats  = ['8 a 11', '12 a 16', 'mas de 16']
df['segmento'] = df['hits_por_periodo'].apply(
    lambda x: 'Heavy user' if x in heavy_cats else 'Light user'
)

seg_stats = (
    df.groupby('segmento')['productividad_score']
    .agg(['mean','median','std','count'])
    .rename(columns={'mean':'Media','median':'Mediana','std':'DE','count':'n'})
    .round(2)
)
print('Estadísticos por segmento:')
print(seg_stats)

# Mann-Whitney U test (no paramétrico)
heavy_scores = df.loc[df['segmento']=='Heavy user', 'productividad_score'].dropna()
light_scores = df.loc[df['segmento']=='Light user', 'productividad_score'].dropna()
u_stat, p_mw = stats.mannwhitneyu(heavy_scores, light_scores, alternative='two-sided')
print(f'\nMann-Whitney U = {u_stat:.1f}, p = {p_mw:.4f}')
print('→', '✅ Diferencia significativa (p < 0.05)' if p_mw < 0.05 else '❌ Sin diferencia significativa')

fig10 = px.violin(df.dropna(subset=['productividad_score','segmento']),
                  x='segmento', y='productividad_score',
                  color='segmento',
                  box=True, points='all',
                  color_discrete_sequence=[ACCENT, '#2cb67d'],
                  title='Distribución de Productividad: Heavy vs Light Users')
fig10.update_layout(**BASE_LAYOUT, height=440)
fig10.show()

In [ ]:
# ─────────────────────────────────────────────
# 7.2 Perfil de usuario: aware vs unaware
#     Consciente = reconoce que afecta su calidad Y concentración
# ─────────────────────────────────────────────
df['perfil'] = df.apply(
    lambda r: 'Consciente'
    if r['afecta_calidad'] == 'si' and r['afecta_concentracion'] == 'si'
    else ('Parcialmente' if 'si' in [r['afecta_calidad'], r['afecta_concentracion']]
                           or 'tal vez' in [r['afecta_calidad'], r['afecta_concentracion']]
          else 'No consciente'), axis=1
)

profile_counts = df['perfil'].value_counts()

fig11 = go.Figure(go.Pie(
    labels=profile_counts.index,
    values=profile_counts.values,
    hole=0.5,
    textinfo='percent+label',
    marker=dict(colors=[PALETTE[0], PALETTE[1], PALETTE[2]])
))
fig11.update_layout(**BASE_LAYOUT,
                    title='Perfil de Autoconciencia del Impacto del Vaping',
                    height=380)
fig11.show()

# Productividad media por perfil
print(df.groupby('perfil')['productividad_score'].agg(['mean','count']).round(2))

## 8. Dashboard Consolidado

In [ ]:
# ─────────────────────────────────────────────
# Dashboard de 6 paneles en una sola figura
# ─────────────────────────────────────────────
from plotly.subplots import make_subplots

fig_dash = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'Distribución de Productividad',
        'Efecto × Productividad Media',
        '% Jornada Perdida (por vida del vape)',
        'Heavy vs Light: Productividad',
        'Autoconciencia del impacto',
        '¿Afecta calidad? × ¿Usa en trabajo?'
    ],
    specs=[
        [{'type':'xy'}, {'type':'xy'}],
        [{'type':'xy'}, {'type':'xy'}],
        [{'type':'pie'},{'type':'xy'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Panel 1: Histograma productividad
fig_dash.add_trace(
    go.Histogram(x=prod, nbinsx=10, marker_color=ACCENT, opacity=0.85, name='Score'),
    row=1, col=1
)

# Panel 2: Bar chart efecto × productividad
fig_dash.add_trace(
    go.Bar(x=prod_by_effect['efecto_vape'], y=prod_by_effect['Media'],
           marker_color=PALETTE[:len(prod_by_effect)],
           text=prod_by_effect['Media'].round(1), textposition='outside',
           name='Media'),
    row=1, col=2
)

# Panel 3: Análisis de sensibilidad
fig_dash.add_trace(
    go.Bar(x=scenario_labels, y=pct_losses, marker_color='#2cb67d',
           text=[f'{p:.0f}%' for p in pct_losses], textposition='outside',
           name='% perdido'),
    row=2, col=1
)

# Panel 4: Box heavy vs light
for seg, color in [('Heavy user', '#ff6b6b'), ('Light user', '#2cb67d')]:
    subset = df.loc[df['segmento'] == seg, 'productividad_score'].dropna()
    fig_dash.add_trace(
        go.Box(y=subset, name=seg, marker_color=color, boxmean=True),
        row=2, col=2
    )

# Panel 5: Pie perfil
fig_dash.add_trace(
    go.Pie(labels=profile_counts.index, values=profile_counts.values,
           hole=0.4, marker=dict(colors=PALETTE[:3])),
    row=3, col=1
)

# Panel 6: Crosstab calidad × trabajo como heatmap
cross2 = pd.crosstab(df['afecta_calidad'], df['uso_trabajo_escuela'])
fig_dash.add_trace(
    go.Heatmap(z=cross2.values, x=cross2.columns.tolist(),
               y=cross2.index.tolist(), colorscale='Viridis',
               showscale=False),
    row=3, col=2
)

fig_dash.update_layout(
    **BASE_LAYOUT,
    title=dict(text='📊 Dashboard: Impacto del Vaping en la Productividad', font_size=20),
    height=1050,
    showlegend=False
)
fig_dash.show()

## 9. Conclusiones

---

### 🔑 Hallazgos principales

| # | Hallazgo | Evidencia |
|---|----------|-----------|
| 1 | **Productividad auto-reportada promedio: ~67%** — los usuarios perciben una reducción significativa | `mean = 6.7/10` |
| 2 | **El mareo es el efecto con mayor impacto negativo en productividad** — menor media del grupo | Análisis por efecto |
| 3 | **Se pierde entre 7% y 71% de la jornada** dependiendo de la intensidad de uso | Modelo de pérdida de tiempo |
| 4 | **Heavy users reportan significativamente menor productividad** (Mann-Whitney p < 0.05) | Sección 7.1 |
| 5 | **La mayoría de usuarios NO es consciente del impacto real** — usan en el trabajo pero niegan efectos | Perfil de autoconciencia + Chi² |
| 6 | **Afectar la concentración y afectar la calidad del trabajo están correlacionadas** (Chi² significativo) | Sección 6.1 |

---

### ⚠️ Limitaciones

- **Sesgo de autoinforme**: los usuarios pueden subestimar impactos negativos.
- **Muestra de conveniencia** (n = 146): no es representativa de toda la población.
- El modelo de pérdida de tiempo asume **comportamiento homogéneo** durante la jornada laboral.
- Variables confusoras (carga de trabajo, edad, tipo de empleo) no fueron controladas.

---

### 🔭 Próximos pasos

- Ampliar la muestra con segmentación por industria/rol laboral.
- Incorporar medidas objetivas de productividad (tareas completadas, errores, etc.).
- Análisis longitudinal: seguimiento de los mismos usuarios antes/después de dejar el vaping.